In [27]:
from pathlib import Path

import pandas as pd
import torch


PROD = False

PATH_PLAYLIST = Path("../data/playlist")
PATH_MODEL = PATH_PLAYLIST / "model_t577K_ep10_v1d3907.pt"
PATH_LOOKUP = PATH_PLAYLIST / "track_lookup.parquet"
PATH_VOCAB = PATH_PLAYLIST / "training_vocab.parquet"


# Loading model

In [28]:
from src.model import Word2Vec


device = torch.device('cpu')

ckpt    = torch.load(PATH_MODEL, map_location=device, weights_only=False)
hparams = ckpt["hparams"]

model = Word2Vec(vocab_size=hparams["vocab_size"], embed_dim=hparams["embed_dim"])
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(model)

Word2Vec(
  (embeddings_in): Embedding(576571, 128, sparse=True)
  (embeddings_out): Embedding(576571, 128, sparse=True)
)


# Loading lookup

In [29]:
vocab = pd.DataFrame(ckpt["vocab"])  # track_rowid, track_id — exactly what was used in training

lookup = pd.read_parquet(
    PATH_LOOKUP,
    columns=["track_rowid", "track_name", "artist_name", "track_popularity"],
)
lookup = lookup.merge(vocab, on="track_rowid", how="inner")
print(f"{len(lookup):,} tracks in lookup (should match vocab size {hparams['vocab_size']:,})")
lookup.head()

576,571 tracks in lookup (should match vocab size 576,571)


,track_rowid,track_name,artist_name,track_popularity,track_id
0,1,The Giver,Chappell Roan,89,0
1,8,SMOKE THE PAIN AWAY,Calvin Harris,75,1
2,12,Relationships,HAIM,72,2
3,13,If Only I Could Wait,Bon Iver,72,3
4,14,Walk Home,Bon Iver,65,4


In [30]:
emb = model.track_embeddings
emb_norm = emb / emb.norm(dim=1, keepdim=True)


def find_neighbours(query: str, k: int = 10, diverse: bool = True) -> pd.DataFrame:
    """Find top-k nearest neighbours by cosine similarity.

    `query` is matched case-insensitively against track_name. If multiple tracks
    match, the most popular one is used.
    If `diverse=True`, at most one track per artist is returned.
    """
    matches = lookup[lookup["track_name"].str.contains(query, case=False, na=False)]
    if matches.empty:
        print(f"No tracks matching '{query}'")
        return pd.DataFrame()

    row = matches.sort_values("track_popularity", ascending=False).iloc[0]
    tid = row["track_id"]
    print(f"Query: {row['track_name']} — {row['artist_name']}  (pop {row['track_popularity']})")

    sims = (emb_norm[tid] @ emb_norm.T).cpu().numpy()

    candidates = lookup.copy()
    candidates["similarity"] = sims[candidates["track_id"].values]
    candidates = candidates[candidates["track_id"] != tid]
    candidates = candidates.sort_values("similarity", ascending=False)
    if diverse:
        candidates = candidates.drop_duplicates(subset="artist_name")
    return candidates.head(k)[["track_name", "artist_name", "track_popularity", "similarity"]]

In [36]:
queries = [
    "Raining Blood",
]

for q in queries:
    display(find_neighbours(q))

Query: Raining Blood — Slayer  (pop 70)


,track_name,artist_name,track_popularity,similarity
21116,South Of Heaven,Slayer,63,0.973357
20958,Harvester Of Sorrow,Metallica,55,0.964363
450392,Hellraiser,Motörhead,46,0.959943
21160,Hellraiser - 30th Anniversary Edition,Ozzy Osbourne,56,0.959765
349165,Symphony Of Destruction - Remastered 2012,Megadeth,70,0.959589
348989,N.I.B. - 2009 Remaster,Black Sabbath,67,0.956594
21010,Am I Demon,Danzig,55,0.948537
495840,Year Zero,Ghost,41,0.948078
32860,Got The Time,Anthrax,59,0.946246
507160,Entre dos tierras,Till Lindemann,30,0.945736
